In [2]:
from pathlib import Path

import numpy as np
import pandas as pd

In [3]:

RAW_MACRO_DIR = Path("../data/raw")
OUTPUT_DIR = Path("../outputs/candidate_features")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Raw macro directory:", RAW_MACRO_DIR.resolve())
print("Output directory:", OUTPUT_DIR.resolve())

Raw macro directory: C:\Users\franc\OneDrive\Geop-Model\data\raw
Output directory: C:\Users\franc\OneDrive\Geop-Model\outputs\candidate_features


In [4]:
# Load raw macro data

dgs2 = pd.read_csv(RAW_MACRO_DIR / "DGS2.csv")
epu = pd.read_csv(RAW_MACRO_DIR / "USEPUINDXD.csv")
vix = pd.read_csv(RAW_MACRO_DIR / "VIXCLS.csv")
eurusd = pd.read_csv(RAW_MACRO_DIR / "DEXUSEU.csv")

In [5]:
# nspect raw macro files

print("DGS2")
display(dgs2.head())
print(dgs2.columns.tolist(), "\n")

print("EPU")
display(epu.head())
print(epu.columns.tolist(), "\n")

print("VIX")
display(vix.head())
print(vix.columns.tolist(), "\n")

print("EUR/USD")
display(eurusd.head())
print(eurusd.columns.tolist())

DGS2


,observation_date,DGS2
0,2017-01-03,1.22
1,2017-01-04,1.24
2,2017-01-05,1.17
3,2017-01-06,1.22
4,2017-01-09,1.21


['observation_date', 'DGS2'] 

EPU


,observation_date,USEPUINDXD
0,2017-01-01,235.10
1,2017-01-02,242.04
2,2017-01-03,89.85
3,2017-01-04,101.99
4,2017-01-05,134.47


['observation_date', 'USEPUINDXD'] 

VIX


,observation_date,VIXCLS
0,2017-01-03,12.85
1,2017-01-04,11.85
2,2017-01-05,11.67
3,2017-01-06,11.32
4,2017-01-09,11.56


['observation_date', 'VIXCLS'] 

EUR/USD


,observation_date,DEXUSEU
0,2017-01-03,1.0416
1,2017-01-04,1.0476
2,2017-01-05,1.0598
3,2017-01-06,1.0560
4,2017-01-09,1.0576


['observation_date', 'DEXUSEU']


In [6]:
# Clean raw macro datasets

def clean_fred_series(df, value_col):
    out = df.copy()
    
    # Standardise date column
    out["observation_date"] = pd.to_datetime(out["observation_date"])
    
    # Convert FRED missing markers / strings to numeric NaN
    out[value_col] = pd.to_numeric(out[value_col], errors="coerce")
    
    # Keep only date + series
    out = out[["observation_date", value_col]]
    
    # Sort chronologically and remove duplicate dates if any
    out = (
        out
        .drop_duplicates(subset="observation_date")
        .sort_values("observation_date")
        .reset_index(drop=True)
    )
    
    return out


dgs2 = clean_fred_series(dgs2, "DGS2")
epu = clean_fred_series(epu, "USEPUINDXD")
vix = clean_fred_series(vix, "VIXCLS")
eurusd = clean_fred_series(eurusd, "DEXUSEU")

In [7]:
# Check cleaned macro series

for name, df_ in {
    "DGS2": dgs2,
    "USEPUINDXD": epu,
    "VIXCLS": vix,
    "DEXUSEU": eurusd
}.items():
    
    print(f"\n{name}")
    print("Shape:", df_.shape)
    print("Date range:",
          df_["observation_date"].min(),
          "to",
          df_["observation_date"].max())
    print("Missing values:")
    print(df_.isna().sum())


DGS2
Shape: (2494, 2)
Date range: 2017-01-03 00:00:00 to 2026-07-24 00:00:00
Missing values:
observation_date      0
DGS2                104
dtype: int64

USEPUINDXD
Shape: (3495, 2)
Date range: 2017-01-01 00:00:00 to 2026-07-27 00:00:00
Missing values:
observation_date    0
USEPUINDXD          0
dtype: int64

VIXCLS
Shape: (2495, 2)
Date range: 2017-01-03 00:00:00 to 2026-07-27 00:00:00
Missing values:
observation_date     0
VIXCLS              60
dtype: int64

DEXUSEU
Shape: (2494, 2)
Date range: 2017-01-03 00:00:00 to 2026-07-24 00:00:00
Missing values:
observation_date      0
DEXUSEU             107
dtype: int64


In [8]:
# Merge raw macro series

macro_raw = (
    eurusd
    .merge(dgs2, on="observation_date", how="outer")
    .merge(epu, on="observation_date", how="outer")
    .merge(vix, on="observation_date", how="outer")
    .sort_values("observation_date")
    .reset_index(drop=True)
)

display(macro_raw.head(10))

print("Shape:", macro_raw.shape)
print("\nDate range:")
print(macro_raw["observation_date"].min(), "to", macro_raw["observation_date"].max())

print("\nMissing values:")
print(macro_raw.isna().sum())

,observation_date,DEXUSEU,DGS2,USEPUINDXD,VIXCLS
0,2017-01-01,NaN,NaN,235.10,NaN
1,2017-01-02,NaN,NaN,242.04,NaN
2,2017-01-03,1.0416,1.22,89.85,12.85
3,2017-01-04,1.0476,1.24,101.99,11.85
4,2017-01-05,1.0598,1.17,134.47,11.67
5,2017-01-06,1.0560,1.22,96.97,11.32
6,2017-01-07,NaN,NaN,110.23,NaN
7,2017-01-08,NaN,NaN,272.01,NaN
8,2017-01-09,1.0576,1.21,121.76,11.56
9,2017-01-10,1.0572,1.19,121.90,11.49


Shape: (3495, 5)

Date range:
2017-01-01 00:00:00 to 2026-07-27 00:00:00

Missing values:
observation_date       0
DEXUSEU             1108
DGS2                1105
USEPUINDXD             0
VIXCLS              1060
dtype: int64


In [9]:
# Restrict to geopolitical research period

START_DATE = "2017-01-01"
END_DATE = "2021-01-08"

macro_raw = macro_raw.loc[
    macro_raw["observation_date"].between(START_DATE, END_DATE)
].copy()

macro_raw = macro_raw.reset_index(drop=True)

print("Shape:", macro_raw.shape)
print(
    "Date range:",
    macro_raw["observation_date"].min(),
    "to",
    macro_raw["observation_date"].max()
)

display(macro_raw.head())
display(macro_raw.tail())

Shape: (1469, 5)
Date range: 2017-01-01 00:00:00 to 2021-01-08 00:00:00


,observation_date,DEXUSEU,DGS2,USEPUINDXD,VIXCLS
0,2017-01-01,NaN,NaN,235.10,NaN
1,2017-01-02,NaN,NaN,242.04,NaN
2,2017-01-03,1.0416,1.22,89.85,12.85
3,2017-01-04,1.0476,1.24,101.99,11.85
4,2017-01-05,1.0598,1.17,134.47,11.67


,observation_date,DEXUSEU,DGS2,USEPUINDXD,VIXCLS
1464,2021-01-04,1.2254,0.11,233.30,26.97
1465,2021-01-05,1.2295,0.13,208.61,25.34
1466,2021-01-06,1.2290,0.14,139.81,25.07
1467,2021-01-07,1.2265,0.14,124.29,22.37
1468,2021-01-08,1.2252,0.14,104.80,21.56


In [10]:
# Paths to LLM annotation outputs

LLM_DATA_DIR = Path("../data/processed")

DEEPSEEK_FILE = LLM_DATA_DIR / "deepseek_outputs.csv"
GEMMA_FILE = LLM_DATA_DIR / "gemma_outputs.csv"

deepseek = pd.read_csv(DEEPSEEK_FILE)
gemma = pd.read_csv(GEMMA_FILE)

In [11]:
# Inspect DeepSeek and Gemma outputs

print("DEEPSEEK")
print("Shape:", deepseek.shape)
print("Columns:")
print(deepseek.columns.tolist())
display(deepseek.head())

print("\nGEMMA")
print("Shape:", gemma.shape)
print("Columns:")
print(gemma.columns.tolist())
display(gemma.head())

DEEPSEEK
Shape: (15278, 13)
Columns:
['tweet_index', 'tweet_id', 'date', 'tweet_snippet', 'trade_score', 'sanctions_score', 'fed_pressure_score', 'reasoning', 'source', 'example_ids', 'model', 'prompt_version', 'created_at_utc']


,tweet_index,tweet_id,date,tweet_snippet,trade_score,sanctions_score,fed_pressure_score,reasoning,source,example_ids,model,prompt_version,created_at_utc
0,0,8.154223e+17,2017-01-01 5:00,"TO ALL AMERICANS-#HappyNewYear &, many blessin...",0,0,0,The tweet is a New Year's greeting without ref...,llm:native_structured_output,A017|A030|A073|A036|A051|A035,deepseek-r1:8b,trump-risk-v2.2-grounded-crashsafe,2026-07-28T23:27:14.544558+00:00
1,1,8.159307e+17,2017-01-02 14:40,"Well, the New Year begins. We will, together, ...",0,0,0,The tweet does not mention any Federal Reserve...,llm:native_structured_output,A040|A068|A078|A075|A022|A093,deepseek-r1:8b,trump-risk-v2.2-grounded-crashsafe,2026-07-28T23:27:22.671079+00:00
2,2,8.159738e+17,2017-01-02 17:31,"Chicago murder rate is record setting - 4,331 ...",0,0,0,The tweet discusses crime statistics and reque...,llm:native_structured_output,A003|A092|A020|A058|A005|A096,deepseek-r1:8b,trump-risk-v2.2-grounded-crashsafe,2026-07-28T23:27:31.742958+00:00
3,3,8.159892e+17,2017-01-02 18:32,"""@CNN just released a book called """"Unpreceden...",0,0,0,The tweet discusses a book release and critiqu...,llm:native_structured_output,A067|A069|A015|A095|A036|A058,deepseek-r1:8b,trump-risk-v2.2-grounded-crashsafe,2026-07-28T23:27:40.130518+00:00
4,4,8.159903e+17,2017-01-02 18:37,Various media outlets and pundits say that I t...,0,0,0,The tweet discusses Trump's political victory ...,llm:native_structured_output,A039|A010|A100|A046|A076|A093,deepseek-r1:8b,trump-risk-v2.2-grounded-crashsafe,2026-07-28T23:27:48.971179+00:00



GEMMA
Shape: (15270, 13)
Columns:
['tweet_index', 'tweet_id', 'date', 'tweet_snippet', 'trade_score', 'sanctions_score', 'fed_pressure_score', 'reasoning', 'source', 'example_ids', 'model', 'prompt_version', 'created_at_utc']


,tweet_index,tweet_id,date,tweet_snippet,trade_score,sanctions_score,fed_pressure_score,reasoning,source,example_ids,model,prompt_version,created_at_utc
0,0,8.154223e+17,2017-01-01 05:00:10,"TO ALL AMERICANS-#HappyNewYear &, many blessin...",0,0,0,The tweet is a general holiday greeting and co...,llm:native_structured_output,A001|A002|A003|A004|A005|A038|A098|A007|A017|A...,gemma4:12b,trump-risk-v2.2-grounded-crashsafe,2026-08-11T11:01:26.945579+00:00
1,1,8.159307e+17,2017-01-02 14:40:10,"Well, the New Year begins. We will, together, ...",0,0,0,The tweet is a general campaign slogan and con...,llm:native_structured_output,A020|A032|A100|A098|A072|A074|A040|A023|A001|A...,gemma4:12b,trump-risk-v2.2-grounded-crashsafe,2026-08-11T11:02:49.999370+00:00
2,2,8.159738e+17,2017-01-02 17:31:17,"Chicago murder rate is record setting - 4,331 ...",0,0,0,The tweet discusses domestic crime and local g...,llm:native_structured_output,A003|A006|A019|A008|A022|A095|A021|A060|A012|A...,gemma4:12b,trump-risk-v2.2-grounded-crashsafe,2026-08-11T11:03:10.359381+00:00
3,3,8.159892e+17,2017-01-02 18:32:29,"""@CNN just released a book called """"Unpreceden...",0,0,0,The tweet discusses a book release by CNN and ...,llm:native_structured_output,A067|A071|A045|A007|A096|A013|A004|A030|A080|A...,gemma4:12b,trump-risk-v2.2-grounded-crashsafe,2026-08-11T11:03:32.138213+00:00
4,4,8.159903e+17,2017-01-02 18:37:10,Various media outlets and pundits say that I t...,0,0,0,The tweet focuses entirely on the candidate's ...,llm:native_structured_output,A078|A016|A092|A036|A014|A007|A099|A008|A062|A...,gemma4:12b,trump-risk-v2.2-grounded-crashsafe,2026-08-11T11:03:51.759213+00:00


In [12]:
# Check likely date columns and missing values

for name, df_ in {
    "DeepSeek": deepseek,
    "Gemma": gemma
}.items():
    
    print(f"\n{name}")
    print("Missing values:")
    display(df_.isna().sum().sort_values(ascending=False).head(15))


DeepSeek
Missing values:


example_ids           98
prompt_version         2
model                  2
source                 2
created_at_utc         2
trade_score            0
tweet_snippet          0
date                   0
tweet_id               0
tweet_index            0
reasoning              0
sanctions_score        0
fed_pressure_score     0
dtype: int64


Gemma
Missing values:


example_ids           96
tweet_id               0
date                   0
tweet_snippet          0
tweet_index            0
trade_score            0
sanctions_score        0
reasoning              0
fed_pressure_score     0
source                 0
model                  0
prompt_version         0
created_at_utc         0
dtype: int64

In [13]:
# Prepare LLM outputs for daily aggregation

SCORE_COLS = [
    "trade_score",
    "sanctions_score",
    "fed_pressure_score"
]

def prepare_llm_scores(df):
    out = df.copy()

    # Convert timestamp to calendar date
    out["date"] = pd.to_datetime(out["date"], errors="coerce")
    out["observation_date"] = out["date"].dt.normalize()

    # Ensure scores are numeric
    for col in SCORE_COLS:
        out[col] = pd.to_numeric(out[col], errors="coerce")

    return out


deepseek = prepare_llm_scores(deepseek)
gemma = prepare_llm_scores(gemma)

In [14]:
for name, df_ in {
    "DeepSeek": deepseek,
    "Gemma": gemma
}.items():

    print(f"\n{name}")
    print("Rows:", len(df_))
    print(
        "Date range:",
        df_["observation_date"].min(),
        "to",
        df_["observation_date"].max()
    )

    print("\nMissing scores:")
    print(df_[SCORE_COLS].isna().sum())

    print("\nScore ranges:")
    display(df_[SCORE_COLS].agg(["min", "max", "mean"]))


DeepSeek
Rows: 15278
Date range: 2017-01-01 00:00:00 to 2021-01-08 00:00:00

Missing scores:
trade_score           0
sanctions_score       0
fed_pressure_score    0
dtype: int64

Score ranges:


,trade_score,sanctions_score,fed_pressure_score
min,0.000000,0.000000,0.000000
max,96.000000,96.000000,99.000000
mean,2.809137,0.930619,0.739626



Gemma
Rows: 15270
Date range: 2017-01-01 00:00:00 to 2021-01-08 00:00:00

Missing scores:
trade_score           0
sanctions_score       0
fed_pressure_score    0
dtype: int64

Score ranges:


,trade_score,sanctions_score,fed_pressure_score
min,0.000000,0.000000,0.000000
max,100.000000,96.000000,99.000000
mean,2.657629,0.980419,0.629731


In [15]:
def create_daily_geopolitical_features(df):

    daily = (
        df.groupby("observation_date")[SCORE_COLS]
        .agg(["sum", "mean", "max"])
    )

    # Flatten MultiIndex column names
    daily.columns = [
        f"{score.replace('_score', '')}_{stat}"
        for score, stat in daily.columns
    ]

    return daily.reset_index()


deepseek_daily = create_daily_geopolitical_features(deepseek)
gemma_daily = create_daily_geopolitical_features(gemma)

In [16]:
print("DEEPSEEK DAILY")
print("Shape:", deepseek_daily.shape)
print(
    "Date range:",
    deepseek_daily["observation_date"].min(),
    "to",
    deepseek_daily["observation_date"].max()
)
display(deepseek_daily.head(10))


print("\nGEMMA DAILY")
print("Shape:", gemma_daily.shape)
print(
    "Date range:",
    gemma_daily["observation_date"].min(),
    "to",
    gemma_daily["observation_date"].max()
)
display(gemma_daily.head(10))

DEEPSEEK DAILY
Shape: (1455, 10)
Date range: 2017-01-01 00:00:00 to 2021-01-08 00:00:00


,observation_date,trade_sum,trade_mean,trade_max,sanctions_sum,sanctions_mean,sanctions_max,fed_pressure_sum,fed_pressure_mean,fed_pressure_max
0,2017-01-01,0,0.000000,0,0,0.0,0,0,0.0,0
1,2017-01-02,50,8.333333,50,60,10.0,40,0,0.0,0
2,2017-01-03,90,10.000000,50,0,0.0,0,0,0.0,0
3,2017-01-04,50,6.250000,50,0,0.0,0,0,0.0,0
4,2017-01-05,40,10.000000,40,0,0.0,0,0,0.0,0
5,2017-01-06,40,4.444444,40,0,0.0,0,0,0.0,0
6,2017-01-07,0,0.000000,0,0,0.0,0,0,0.0,0
7,2017-01-08,0,0.000000,0,0,0.0,0,0,0.0,0
8,2017-01-09,105,21.000000,65,25,5.0,25,0,0.0,0
9,2017-01-10,0,0.000000,0,0,0.0,0,0,0.0,0



GEMMA DAILY
Shape: (1455, 10)
Date range: 2017-01-01 00:00:00 to 2021-01-08 00:00:00


,observation_date,trade_sum,trade_mean,trade_max,sanctions_sum,sanctions_mean,sanctions_max,fed_pressure_sum,fed_pressure_mean,fed_pressure_max
0,2017-01-01,0,0.000000,0,0,0.0,0,0,0.0,0
1,2017-01-02,63,10.500000,63,18,3.0,18,0,0.0,0
2,2017-01-03,94,10.444444,84,0,0.0,0,0,0.0,0
3,2017-01-04,25,3.125000,25,0,0.0,0,0,0.0,0
4,2017-01-05,68,17.000000,68,0,0.0,0,0,0.0,0
5,2017-01-06,30,3.333333,30,0,0.0,0,0,0.0,0
6,2017-01-07,0,0.000000,0,0,0.0,0,0,0.0,0
7,2017-01-08,0,0.000000,0,0,0.0,0,0,0.0,0
8,2017-01-09,35,7.000000,20,0,0.0,0,0,0.0,0
9,2017-01-10,0,0.000000,0,0,0.0,0,0,0.0,0


In [17]:
# Generate geopolitical candidate features

BASE_GEO_COLS = [
    "trade_sum", "trade_mean", "trade_max",
    "sanctions_sum", "sanctions_mean", "sanctions_max",
    "fed_pressure_sum", "fed_pressure_mean", "fed_pressure_max"
]

GEO_LAGS = [1, 2, 3, 5]
GEO_WINDOWS = [3, 5, 10]


def create_geo_candidate_features(daily_df):
    out = daily_df.copy()
    out = out.sort_values("observation_date").reset_index(drop=True)

    for col in BASE_GEO_COLS:

        # First change
        out[f"{col}_diff"] = out[col].diff()

        # Calendar-day lags
        for lag in GEO_LAGS:
            out[f"{col}_lag{lag}"] = out[col].shift(lag)

        # Moving averages
        for window in GEO_WINDOWS:
            out[f"{col}_ma{window}"] = (
                out[col]
                .rolling(window=window, min_periods=window)
                .mean()
            )

    return out


deepseek_candidates = create_geo_candidate_features(deepseek_daily)
gemma_candidates = create_geo_candidate_features(gemma_daily)

In [18]:
# Inspect geopolitical candidate sets

for name, df_ in {
    "DeepSeek": deepseek_candidates,
    "Gemma": gemma_candidates
}.items():

    print(f"\n{name}")
    print("Shape:", df_.shape)
    print("Number of geopolitical predictors:", df_.shape[1] - 1)

    print("\nFirst 20 columns:")
    print(df_.columns[:20].tolist())

    print("\nMissing values caused by transformations:")
    print(df_.isna().sum().sum())


DeepSeek
Shape: (1455, 82)
Number of geopolitical predictors: 81

First 20 columns:
['observation_date', 'trade_sum', 'trade_mean', 'trade_max', 'sanctions_sum', 'sanctions_mean', 'sanctions_max', 'fed_pressure_sum', 'fed_pressure_mean', 'fed_pressure_max', 'trade_sum_diff', 'trade_sum_lag1', 'trade_sum_lag2', 'trade_sum_lag3', 'trade_sum_lag5', 'trade_sum_ma3', 'trade_sum_ma5', 'trade_sum_ma10', 'trade_mean_diff', 'trade_mean_lag1']

Missing values caused by transformations:
243

Gemma
Shape: (1455, 82)
Number of geopolitical predictors: 81

First 20 columns:
['observation_date', 'trade_sum', 'trade_mean', 'trade_max', 'sanctions_sum', 'sanctions_mean', 'sanctions_max', 'fed_pressure_sum', 'fed_pressure_mean', 'fed_pressure_max', 'trade_sum_diff', 'trade_sum_lag1', 'trade_sum_lag2', 'trade_sum_lag3', 'trade_sum_lag5', 'trade_sum_ma3', 'trade_sum_ma5', 'trade_sum_ma10', 'trade_mean_diff', 'trade_mean_lag1']

Missing values caused by transformations:
243


In [19]:
# Create financial-market modelling grid

macro_features = macro_raw.loc[
    macro_raw["DEXUSEU"].notna()
].copy()

macro_features = (
    macro_features
    .sort_values("observation_date")
    .reset_index(drop=True)
)

print("Financial observations:", len(macro_features))
print(
    "Date range:",
    macro_features["observation_date"].min(),
    "to",
    macro_features["observation_date"].max()
)

print("\nMissing values on EUR/USD observation dates:")
print(
    macro_features[
        ["DEXUSEU", "DGS2", "USEPUINDXD", "VIXCLS"]
    ].isna().sum()
)

Financial observations: 1002
Date range: 2017-01-03 00:00:00 to 2021-01-08 00:00:00

Missing values on EUR/USD observation dates:
DEXUSEU       0
DGS2          4
USEPUINDXD    0
VIXCLS        4
dtype: int64


In [20]:
# Keep complete financial-market observations

MACRO_RAW_COLS = [
    "DEXUSEU",
    "DGS2",
    "USEPUINDXD",
    "VIXCLS"
]

macro_features = (
    macro_features
    .dropna(subset=MACRO_RAW_COLS)
    .copy()
    .reset_index(drop=True)
)

print("Complete financial observations:", len(macro_features))

print(
    "Date range:",
    macro_features["observation_date"].min(),
    "to",
    macro_features["observation_date"].max()
)

print("\nMissing values:")
print(macro_features[MACRO_RAW_COLS].isna().sum())

Complete financial observations: 998
Date range: 2017-01-03 00:00:00 to 2021-01-08 00:00:00

Missing values:
DEXUSEU       0
DGS2          0
USEPUINDXD    0
VIXCLS        0
dtype: int64


In [21]:
# EUR/USD target

macro_features["DEXUSEU_logreturn"] = (
    np.log(macro_features["DEXUSEU"])
    .diff()
)

In [22]:
# Generate macro candidate features

MACRO_BASE_COLS = [
    "DGS2",
    "USEPUINDXD",
    "VIXCLS"
]

MACRO_LAGS = [1, 2, 3, 5]
MACRO_WINDOWS = [3, 5, 10]


for col in MACRO_BASE_COLS:

    # Current change
    macro_features[f"{col}_diff"] = (
        macro_features[col].diff()
    )

    # Lagged levels
    for lag in MACRO_LAGS:
        macro_features[f"{col}_lag{lag}"] = (
            macro_features[col].shift(lag)
        )

    # Lagged changes
    for lag in MACRO_LAGS:
        macro_features[f"{col}_diff_lag{lag}"] = (
            macro_features[f"{col}_diff"].shift(lag)
        )

    # Moving averages of levels
    for window in MACRO_WINDOWS:
        macro_features[f"{col}_ma{window}"] = (
            macro_features[col]
            .rolling(
                window=window,
                min_periods=window
            )
            .mean()
        )

In [23]:
# Lagged EUR/USD returns

for lag in MACRO_LAGS:
    macro_features[f"DEXUSEU_logreturn_lag{lag}"] = (
        macro_features["DEXUSEU_logreturn"].shift(lag)
    )

In [24]:
# Check macro candidate dataset

print("Shape:", macro_features.shape)

print("\nMissing values after feature construction:")
print(
    macro_features
    .isna()
    .sum()
    .sort_values(ascending=False)
    .head(20)
)

print("\nFirst date with complete candidate information:")

complete_macro = macro_features.dropna()

print(complete_macro["observation_date"].min())
print("Complete rows:", len(complete_macro))

Shape: (998, 46)

Missing values after feature construction:
USEPUINDXD_ma10           9
DGS2_ma10                 9
VIXCLS_ma10               9
USEPUINDXD_diff_lag5      6
DEXUSEU_logreturn_lag5    6
VIXCLS_diff_lag5          6
DGS2_diff_lag5            6
USEPUINDXD_lag5           5
DGS2_lag5                 5
VIXCLS_lag5               5
VIXCLS_ma5                4
VIXCLS_diff_lag3          4
DEXUSEU_logreturn_lag3    4
DGS2_diff_lag3            4
DGS2_ma5                  4
USEPUINDXD_diff_lag3      4
USEPUINDXD_ma5            4
VIXCLS_lag3               3
VIXCLS_diff_lag2          3
DEXUSEU_logreturn_lag2    3
dtype: int64

First date with complete candidate information:
2017-01-17 00:00:00
Complete rows: 989


In [25]:
# Merge macro and geopolitical candidate features

deepseek_full = macro_features.merge(
    deepseek_candidates,
    on="observation_date",
    how="left"
)

gemma_full = macro_features.merge(
    gemma_candidates,
    on="observation_date",
    how="left"
)

print("DeepSeek:", deepseek_full.shape)
print("Gemma:", gemma_full.shape)

DeepSeek: (998, 127)
Gemma: (998, 127)


In [26]:
# Check merged datasets

for name, df_ in {
    "DeepSeek": deepseek_full,
    "Gemma": gemma_full
}.items():

    print(f"\n{name}")
    print("Shape:", df_.shape)

    print(
        "Date range:",
        df_["observation_date"].min(),
        "to",
        df_["observation_date"].max()
    )

    print("Total missing values:", df_.isna().sum().sum())

    print("\nColumns with most missing values:")
    print(
        df_.isna()
        .sum()
        .sort_values(ascending=False)
        .head(15)
    )


DeepSeek
Shape: (998, 127)
Date range: 2017-01-03 00:00:00 to 2021-01-08 00:00:00
Total missing values: 808

Columns with most missing values:
trade_sum_ma10            12
fed_pressure_max_ma10     12
fed_pressure_mean_ma10    12
sanctions_sum_ma10        12
sanctions_mean_ma10       12
trade_max_ma10            12
trade_mean_ma10           12
sanctions_max_ma10        12
fed_pressure_sum_ma10     12
trade_sum_lag5            10
sanctions_sum_lag5        10
fed_pressure_mean_lag5    10
fed_pressure_max_lag5     10
fed_pressure_sum_lag5     10
sanctions_max_lag5        10
dtype: int64

Gemma
Shape: (998, 127)
Date range: 2017-01-03 00:00:00 to 2021-01-08 00:00:00
Total missing values: 808

Columns with most missing values:
trade_sum_ma10            12
fed_pressure_max_ma10     12
fed_pressure_mean_ma10    12
sanctions_sum_ma10        12
sanctions_mean_ma10       12
trade_max_ma10            12
trade_mean_ma10           12
sanctions_max_ma10        12
fed_pressure_sum_ma10     12
trade_

In [27]:
# Define target

TARGET = "DEXUSEU_logreturn"

# Columns that are not candidate predictors
NON_PREDICTORS = [
    "observation_date",
    "DEXUSEU",              # contemporaneous EUR/USD level
    TARGET                   # target itself
]

In [28]:
# Find common complete modelling dates

deepseek_complete_dates = deepseek_full.dropna()["observation_date"]
gemma_complete_dates = gemma_full.dropna()["observation_date"]

common_dates = (
    pd.Index(deepseek_complete_dates)
    .intersection(pd.Index(gemma_complete_dates))
    .sort_values()
)

print("DeepSeek complete dates:", len(deepseek_complete_dates))
print("Gemma complete dates:", len(gemma_complete_dates))
print("Common complete dates:", len(common_dates))

print(
    "Common date range:",
    common_dates.min(),
    "to",
    common_dates.max()
)

DeepSeek complete dates: 982
Gemma complete dates: 982
Common complete dates: 982
Common date range: 2017-01-17 00:00:00 to 2021-01-08 00:00:00


In [29]:
# Restrict DeepSeek and Gemma to identical sample

deepseek_model = (
    deepseek_full[
        deepseek_full["observation_date"].isin(common_dates)
    ]
    .copy()
    .reset_index(drop=True)
)

gemma_model = (
    gemma_full[
        gemma_full["observation_date"].isin(common_dates)
    ]
    .copy()
    .reset_index(drop=True)
)

print("DeepSeek:", deepseek_model.shape)
print("Gemma:", gemma_model.shape)

print("\nMissing values:")
print("DeepSeek:", deepseek_model.isna().sum().sum())
print("Gemma:", gemma_model.isna().sum().sum())

print(
    "\nIdentical dates:",
    deepseek_model["observation_date"].equals(
        gemma_model["observation_date"]
    )
)

DeepSeek: (982, 127)
Gemma: (982, 127)

Missing values:
DeepSeek: 0
Gemma: 0

Identical dates: True


In [30]:
# Candidate predictor matrices

predictor_cols = [
    col for col in deepseek_model.columns
    if col not in NON_PREDICTORS
]

X_deepseek = deepseek_model[predictor_cols].copy()
X_gemma = gemma_model[predictor_cols].copy()

y_deepseek = deepseek_model[TARGET].copy()
y_gemma = gemma_model[TARGET].copy()

print("DeepSeek X:", X_deepseek.shape)
print("Gemma X:", X_gemma.shape)

print("DeepSeek y:", y_deepseek.shape)
print("Gemma y:", y_gemma.shape)

print(
    "\nTargets identical:",
    np.allclose(y_deepseek, y_gemma)
)

print("\nNumber of candidate predictors:", len(predictor_cols))

DeepSeek X: (982, 124)
Gemma X: (982, 124)
DeepSeek y: (982,)
Gemma y: (982,)

Targets identical: True

Number of candidate predictors: 124


In [31]:
# Check predictors with zero variance

constant_deepseek = X_deepseek.columns[
    X_deepseek.nunique() <= 1
].tolist()

constant_gemma = X_gemma.columns[
    X_gemma.nunique() <= 1
].tolist()

print("DeepSeek constant predictors:")
print(constant_deepseek)

print("\nGemma constant predictors:")
print(constant_gemma)

DeepSeek constant predictors:
[]

Gemma constant predictors:
[]


In [32]:
# Find exact duplicate columns

def find_duplicate_columns(df):
    duplicates = []

    cols = df.columns

    for i in range(len(cols)):
        for j in range(i + 1, len(cols)):
            if df[cols[i]].equals(df[cols[j]]):
                duplicates.append((cols[i], cols[j]))

    return duplicates


duplicate_deepseek = find_duplicate_columns(X_deepseek)
duplicate_gemma = find_duplicate_columns(X_gemma)

print("DeepSeek exact duplicates:")
print(duplicate_deepseek)

print("\nGemma exact duplicates:")
print(duplicate_gemma)

DeepSeek exact duplicates:
[]

Gemma exact duplicates:
[]


In [33]:
# Find highly correlated predictor pairs

def high_correlation_pairs(df, threshold=0.995):
    corr = df.corr().abs()

    upper = corr.where(
        np.triu(
            np.ones(corr.shape),
            k=1
        ).astype(bool)
    )

    pairs = []

    for col in upper.columns:
        for row in upper.index:
            value = upper.loc[row, col]

            if pd.notna(value) and value >= threshold:
                pairs.append((row, col, value))

    return sorted(
        pairs,
        key=lambda x: x[2],
        reverse=True
    )


high_corr_deepseek = high_correlation_pairs(
    X_deepseek,
    threshold=0.995
)

high_corr_gemma = high_correlation_pairs(
    X_gemma,
    threshold=0.995
)

print("DeepSeek pairs >= 0.995:")
for pair in high_corr_deepseek:
    print(pair)

print("\nGemma pairs >= 0.995:")
for pair in high_corr_gemma:
    print(pair)

DeepSeek pairs >= 0.995:
('DGS2_lag1', 'DGS2_ma3', np.float64(0.9998174727007422))
('DGS2_ma3', 'DGS2_ma5', np.float64(0.9997792079618623))
('DGS2_lag2', 'DGS2_ma5', np.float64(0.9997091407878996))
('DGS2', 'DGS2_ma3', np.float64(0.9995876861861087))
('DGS2_lag2', 'DGS2_ma3', np.float64(0.9995846897430499))
('DGS2_lag1', 'DGS2_ma5', np.float64(0.9995466281314538))
('DGS2_lag3', 'DGS2_ma5', np.float64(0.9995442309974714))
('DGS2_lag5', 'DGS2_ma10', np.float64(0.9994026084454479))
('DGS2_ma5', 'DGS2_ma10', np.float64(0.9992492325625656))
('DGS2', 'DGS2_lag1', np.float64(0.9992237497340395))
('DGS2_lag1', 'DGS2_lag2', np.float64(0.9992187037256495))
('DGS2_lag2', 'DGS2_lag3', np.float64(0.9992179491725184))
('DGS2_lag3', 'DGS2_ma10', np.float64(0.9992125592078746))
('DGS2', 'DGS2_ma5', np.float64(0.9990727814302698))
('DGS2_lag2', 'DGS2_ma10', np.float64(0.9988195161472452))
('DGS2_lag3', 'DGS2_ma3', np.float64(0.9988187913097701))
('DGS2', 'DGS2_lag2', np.float64(0.9985276052691113))
('D

In [34]:
# Final candidate feature datasets

final_cols = [
    "observation_date",
    TARGET
] + predictor_cols

candidate_deepseek = deepseek_model[final_cols].copy()
candidate_gemma = gemma_model[final_cols].copy()

print("DeepSeek final shape:", candidate_deepseek.shape)
print("Gemma final shape:", candidate_gemma.shape)

print("\nExpected:")
print("982 observations")
print("126 columns = date + target + 124 predictors")

print("\nMissing values:")
print("DeepSeek:", candidate_deepseek.isna().sum().sum())
print("Gemma:", candidate_gemma.isna().sum().sum())

print(
    "\nIdentical dates:",
    candidate_deepseek["observation_date"].equals(
        candidate_gemma["observation_date"]
    )
)

print(
    "Identical targets:",
    np.allclose(
        candidate_deepseek[TARGET],
        candidate_gemma[TARGET]
    )
)

DeepSeek final shape: (982, 126)
Gemma final shape: (982, 126)

Expected:
982 observations
126 columns = date + target + 124 predictors

Missing values:
DeepSeek: 0
Gemma: 0

Identical dates: True
Identical targets: True


In [35]:
# Save final candidate datasets

deepseek_path = OUTPUT_DIR / "candidate_features_deepseek.csv"
gemma_path = OUTPUT_DIR / "candidate_features_gemma.csv"

candidate_deepseek.to_csv(
    deepseek_path,
    index=False
)

candidate_gemma.to_csv(
    gemma_path,
    index=False
)

print("Saved:")
print(deepseek_path)
print(gemma_path)

Saved:
..\outputs\candidate_features\candidate_features_deepseek.csv
..\outputs\candidate_features\candidate_features_gemma.csv


In [36]:
# Create feature manifest

def classify_feature(col):

    if col.startswith("DEXUSEU_logreturn_lag"):
        source = "EURUSD"
    elif col.startswith(("DGS2", "USEPUINDXD", "VIXCLS")):
        source = "Macro"
    else:
        source = "Geopolitical"

    if "_diff_lag" in col:
        transformation = "lagged_difference"
    elif col.endswith("_diff"):
        transformation = "difference"
    elif "_lag" in col:
        transformation = "lag"
    elif "_ma" in col:
        transformation = "moving_average"
    else:
        transformation = "level_or_daily_aggregation"

    return source, transformation


manifest_rows = []

for col in predictor_cols:

    source, transformation = classify_feature(col)

    manifest_rows.append({
        "feature": col,
        "source": source,
        "transformation": transformation
    })


feature_manifest = pd.DataFrame(manifest_rows)

display(feature_manifest)

print("\nFeatures by source:")
print(feature_manifest["source"].value_counts())

print("\nFeatures by transformation:")
print(feature_manifest["transformation"].value_counts())

,feature,source,transformation
0,DGS2,Macro,level_or_daily_aggregation
1,USEPUINDXD,Macro,level_or_daily_aggregation
2,VIXCLS,Macro,level_or_daily_aggregation
3,DGS2_diff,Macro,difference
4,DGS2_lag1,Macro,lag
...,...,...,...
119,fed_pressure_max_lag3,Geopolitical,lag
120,fed_pressure_max_lag5,Geopolitical,lag
121,fed_pressure_max_ma3,Geopolitical,moving_average
122,fed_pressure_max_ma5,Geopolitical,moving_average



Features by source:
source
Geopolitical    81
Macro           39
EURUSD           4
Name: count, dtype: int64

Features by transformation:
transformation
lag                           52
moving_average                39
difference                    12
lagged_difference             12
level_or_daily_aggregation     9
Name: count, dtype: int64


In [37]:
# Cell 36 — Save feature manifest

manifest_path = OUTPUT_DIR / "candidate_feature_manifest.csv"

feature_manifest.to_csv(
    manifest_path,
    index=False
)

print("Saved:", manifest_path)

Saved: ..\outputs\candidate_features\candidate_feature_manifest.csv


In [38]:
# Final sanity check

print("=" * 50)
print("CANDIDATE FEATURE DATASET COMPLETE")
print("=" * 50)

print(f"Observations: {len(candidate_deepseek)}")
print(f"Candidate predictors: {len(predictor_cols)}")

print(
    "Sample:",
    candidate_deepseek["observation_date"].min().date(),
    "to",
    candidate_deepseek["observation_date"].max().date()
)

print("\nTarget:")
print(TARGET)

print("\nDeepSeek dataset:")
print(candidate_deepseek.shape)

print("\nGemma dataset:")
print(candidate_gemma.shape)

print("\nFeature groups:")
print(feature_manifest["source"].value_counts())

print("\nMissing values:")
print("DeepSeek:", candidate_deepseek.isna().sum().sum())
print("Gemma:", candidate_gemma.isna().sum().sum())

CANDIDATE FEATURE DATASET COMPLETE
Observations: 982
Candidate predictors: 124
Sample: 2017-01-17 to 2021-01-08

Target:
DEXUSEU_logreturn

DeepSeek dataset:
(982, 126)

Gemma dataset:
(982, 126)

Feature groups:
source
Geopolitical    81
Macro           39
EURUSD           4
Name: count, dtype: int64

Missing values:
DeepSeek: 0
Gemma: 0
